# 06 — Bayesian Thinking

Bayesian statistics provides a principled framework for updating beliefs with evidence. Unlike frequentist statistics (fixed parameters, varying data), Bayesian statistics treats parameters as probability distributions — and updates them as data arrives.

**Topics:**
- Frequentist vs Bayesian philosophy
- Prior, Likelihood, and Posterior
- Conjugate priors (Beta-Binomial)
- Bayesian updating (sequential learning)
- Credible intervals vs confidence intervals
- Metropolis-Hastings MCMC from scratch
- PyMC intro (if installed)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

## 1. The Bayesian Framework

**Posterior ∝ Likelihood × Prior**

```
P(θ | data) = P(data | θ) × P(θ) / P(data)
               Likelihood    Prior    Normalizer
```

We start with a **prior** belief about θ, observe data, compute the **likelihood** of that data given θ, and update to get the **posterior** — our refined belief.

In [ ]:
# Canonical example: estimating the bias of a coin
# θ = P(heads) — unknown, between 0 and 1
# Prior: Beta(α, β) — conjugate prior for binomial likelihood
# After n flips with h heads: Posterior = Beta(α+h, β+n-h)

theta = np.linspace(0, 1, 1000)

# True bias (unknown to the model)
true_theta = 0.7  # biased coin: 70% heads

# Start with uniform prior: Beta(1,1) = "I know nothing"
alpha_prior, beta_prior = 1, 1

observations = [1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1]  # 14H, 6T
print(f'True θ = {true_theta}, Observations: {sum(observations)} heads out of {len(observations)} flips')

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

show_at = [0, 1, 3, 7, 14, 20]
alpha_post, beta_post = alpha_prior, beta_prior

for plot_idx, n_obs in enumerate(show_at):
    ax = axes[plot_idx]
    
    if n_obs == 0:
        a, b = alpha_prior, beta_prior
        title = f'Prior: Beta({a},{b})'
    else:
        heads = sum(observations[:n_obs])
        tails = n_obs - heads
        a = alpha_prior + heads
        b = beta_prior + tails
        title = f'After {n_obs} flips ({heads}H,{tails}T)\nBeta({a},{b})'
    
    posterior = stats.beta.pdf(theta, a, b)
    ax.plot(theta, posterior, 'b-', lw=2)
    ax.fill_between(theta, posterior, alpha=0.3, color='steelblue')
    ax.axvline(true_theta, color='red', ls='--', lw=2, label=f'True θ={true_theta}')
    ax.axvline(a/(a+b), color='green', ls=':', lw=2, label=f'MAP={a/(a+b):.2f}')
    
    # 95% credible interval
    ci_low, ci_high = stats.beta.ppf([0.025, 0.975], a, b)
    ax.fill_between(theta, posterior, 
                    where=(theta>=ci_low) & (theta<=ci_high), 
                    alpha=0.5, color='darkorange', label=f'95% CI [{ci_low:.2f},{ci_high:.2f}]')
    
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('θ = P(heads)')
    ax.set_ylabel('Posterior density')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Bayesian Updating: Coin Bias Estimation (Beta-Binomial Conjugate)', 
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 2. Conjugate Priors — Closed-Form Updates

In [ ]:
print('Conjugate Prior Table')
print('='*65)
print(f'{"Likelihood":<20}{"Prior":<20}{"Posterior"}')
print('-'*65)
conjugates = [
    ('Bernoulli/Binomial', 'Beta(α,β)', 'Beta(α+h, β+n-h)'),
    ('Poisson', 'Gamma(α,β)', 'Gamma(α+Σx, β+n)'),
    ('Normal (known σ)', 'Normal(μ₀,σ₀²)', 'Normal(μₙ,σₙ²)'),
    ('Multinomial', 'Dirichlet(α)', 'Dirichlet(α+counts)'),
    ('Exponential', 'Gamma(α,β)', 'Gamma(α+n, β+Σx)'),
]
for likelihood, prior, posterior in conjugates:
    print(f'{likelihood:<20}{prior:<20}{posterior}')

# Normal-Normal conjugate: estimating mean with known variance
print('\n--- Normal-Normal Conjugate Demo ---')
true_mu = 5.0
sigma_known = 2.0  # assume we know the population std
n_data = 20
data = np.random.normal(true_mu, sigma_known, n_data)

# Prior: N(mu_0, sigma_0^2)
mu_0, sigma_0 = 0.0, 10.0  # vague prior

# Posterior parameters (Normal-Normal conjugate)
sigma_0_sq = sigma_0**2
sigma_sq = sigma_known**2
n = len(data)
x_bar = data.mean()

# Precision-weighted update
sigma_n_sq = 1 / (1/sigma_0_sq + n/sigma_sq)
mu_n = sigma_n_sq * (mu_0/sigma_0_sq + n*x_bar/sigma_sq)

theta_plot = np.linspace(-5, 15, 500)
prior_dist = stats.norm.pdf(theta_plot, mu_0, sigma_0)
posterior_dist = stats.norm.pdf(theta_plot, mu_n, np.sqrt(sigma_n_sq))
likelihood_dist = stats.norm.pdf(theta_plot, x_bar, sigma_known/np.sqrt(n))

plt.figure(figsize=(10, 5))
plt.plot(theta_plot, prior_dist / prior_dist.max(), 'b--', lw=2, label=f'Prior N({mu_0},{sigma_0})')
plt.plot(theta_plot, likelihood_dist / likelihood_dist.max(), 'g-', lw=2, 
         label=f'Likelihood (data: x̄={x_bar:.2f})')
plt.plot(theta_plot, posterior_dist / posterior_dist.max(), 'r-', lw=2, 
         label=f'Posterior N({mu_n:.2f},{np.sqrt(sigma_n_sq):.2f})')
plt.axvline(true_mu, color='black', ls=':', lw=2, label=f'True μ={true_mu}')
plt.xlabel('μ'); plt.ylabel('Density (normalized)')
plt.title('Normal-Normal Conjugate: Posterior = Prior × Likelihood')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Metropolis-Hastings MCMC from Scratch

When conjugate priors don't exist, we use Markov Chain Monte Carlo (MCMC) to sample from the posterior numerically.

In [ ]:
# Problem: Estimate the mean of a distribution given data
# with a non-conjugate setup

# True parameters
true_mu_mcmc = 3.0
true_sigma_mcmc = 1.5
n_data_mcmc = 30
observed_data = np.random.normal(true_mu_mcmc, true_sigma_mcmc, n_data_mcmc)

def log_likelihood(mu, data, sigma=1.5):
    """Log probability of data given parameter mu."""
    return np.sum(stats.norm.logpdf(data, loc=mu, scale=sigma))

def log_prior(mu, prior_mu=0, prior_sigma=5):
    """Log probability of parameter (weakly informative prior)."""
    return stats.norm.logpdf(mu, prior_mu, prior_sigma)

def log_posterior(mu, data):
    return log_likelihood(mu, data) + log_prior(mu)

# Metropolis-Hastings Algorithm
n_samples = 10000
proposal_std = 0.5  # proposal distribution width
chain = np.zeros(n_samples)
chain[0] = 0.0  # starting point
accepted = 0

for i in range(1, n_samples):
    current = chain[i-1]
    # Propose a new value
    proposal = current + np.random.normal(0, proposal_std)
    
    # Acceptance ratio (in log space for numerical stability)
    log_accept = log_posterior(proposal, observed_data) - log_posterior(current, observed_data)
    
    # Accept or reject
    if np.log(np.random.uniform()) < log_accept:
        chain[i] = proposal
        accepted += 1
    else:
        chain[i] = current

burn_in = 2000
posterior_samples = chain[burn_in:]
print(f'Acceptance rate: {accepted/n_samples:.1%} (ideal: 20-50%)')
print(f'True μ: {true_mu_mcmc}')
print(f'Posterior mean: {posterior_samples.mean():.3f}')
print(f'95% Credible Interval: [{np.percentile(posterior_samples, 2.5):.3f}, {np.percentile(posterior_samples, 97.5):.3f}]')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Trace plot
axes[0].plot(chain[:500], 'b-', lw=0.5, alpha=0.7)
axes[0].axvline(burn_in, color='red', ls='--', label='Burn-in end')
axes[0].axhline(true_mu_mcmc, color='green', ls='--', label=f'True μ={true_mu_mcmc}')
axes[0].set_title('MCMC Trace (first 500 steps)'); axes[0].set_xlabel('Step')
axes[0].set_ylabel('μ'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Posterior distribution
axes[1].hist(posterior_samples, bins=60, density=True, color='steelblue', alpha=0.7)
axes[1].axvline(true_mu_mcmc, color='red', ls='--', lw=2, label=f'True μ={true_mu_mcmc}')
axes[1].axvline(posterior_samples.mean(), color='green', ls='--', lw=2, 
                label=f'Posterior mean={posterior_samples.mean():.2f}')
ci = np.percentile(posterior_samples, [2.5, 97.5])
axes[1].axvspan(ci[0], ci[1], alpha=0.2, color='orange', label=f'95% CI [{ci[0]:.2f},{ci[1]:.2f}]')
axes[1].set_title('Posterior Distribution (MCMC)'); axes[1].set_xlabel('μ')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

# Autocorrelation
lags = np.arange(0, 50)
autocorr = [np.corrcoef(posterior_samples[:-lag], posterior_samples[lag:])[0,1] if lag > 0 else 1.0
            for lag in lags]
axes[2].plot(lags, autocorr, 'o-', markersize=4)
axes[2].axhline(0, color='k', ls='--')
axes[2].set_title('Autocorrelation of MCMC Chain'); axes[2].set_xlabel('Lag')
axes[2].set_ylabel('Autocorrelation'); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 4. Bayesian vs Frequentist: Key Differences

| Aspect | Frequentist | Bayesian |
|--------|------------|----------|
| Parameters | Fixed, unknown constants | Random variables with distributions |
| Probability | Long-run frequency | Degree of belief |
| Inference | p-values, confidence intervals | Posterior distributions, credible intervals |
| Prior knowledge | Ignored | Explicitly incorporated |
| Small data | Struggles | Can use informative priors |
| Interpretability | "Prob of data given H₀" | "Prob of hypothesis given data" |

**Credible interval vs Confidence interval:**
- **95% CI (frequentist):** If we repeated the experiment many times, 95% of CIs would contain the true parameter
- **95% Credible interval (Bayesian):** Given the data we observed, there's a 95% probability the parameter lies in this range

The Bayesian interpretation is what people intuitively want from a confidence interval!

## Summary

| Concept | Formula | Implementation |
|---------|---------|---------------|
| Bayes' theorem | P(θ\|D) ∝ P(D\|θ)·P(θ) | Manual or PyMC |
| Beta-Binomial | Beta(α+h, β+n-h) | `scipy.stats.beta` |
| Credible interval | 2.5th to 97.5th percentile of posterior | `np.percentile` |
| MCMC | Sample posterior numerically | PyMC, Stan, NumPyro |

**Next:** [07_information_theory.ipynb](07_information_theory.ipynb)